In [18]:
from pathlib import Path
import utils as ut
import matplotlib.pyplot as plt
import matplotlib

import pandas as pd
pd.set_option('display.max_columns', None)

from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.wcs import WCS

In [19]:
CUT = True

In [20]:
cwd = Path.cwd()
table_dir = cwd / 'data' / 'tables'
all_dir =  cwd / 'data' / 'tables_all'
mosaic_dir = cwd / 'data' / 'mosaics'
cutout_dir = cwd / 'data' / 'cutouts'

In [21]:
table_dfs = {}
all_dfs = {}
for file in table_dir.iterdir():
    table_dfs[file.stem] = pd.read_csv(file).sort_values(by="RA")
for file in all_dir.iterdir():
    all_dfs[file.stem] = pd.read_csv(file).sort_values(by="RA")

headers = ut.fits_df(mosaic_dir)
headers = headers[['FILENAME'] + [col for col in headers.columns if col != 'FILENAME']]

In [22]:
if CUT: 
    with_two = []
    size = 3

    stamp = u.Quantity([size, size], u.arcsec)

    for _, row in headers.iterrows():
        path = row['FILEPATH']
        catalogs = [('ceers', 'ceers'),
                    ('cosmos', 'primer_cosmos'),
                    ('uds', 'primer_uds')]
        bands = ['f150w', 'f277w', 'f444w']

        band = None
        catalog = None

        for cat, b in zip(catalogs, bands):
            if cat[0] in path.name:
                catalog = cat[1]
            if b in path.name:
                band = b

        table = table_dfs[catalog]
        all_table = all_dfs[catalog + '_all']

        hdul = fits.open(path)
        data = hdul[0].data
        wcs = WCS(hdul[0].header)

        for _, row in table.iterrows():

            arcsecs = (size * u.arcsec) + (0.25 * u.arcsec)
            ra = row['RA'] * u.deg
            dec = row['DEC'] * u.deg
            ramask = (all_table['RA'] < (ra + arcsecs).value) & (all_table['RA'] > (ra - arcsecs).value)
            decmask = (all_table['DEC'] < (dec + arcsecs).value) & (all_table['DEC'] > (dec - arcsecs).value)
            num_objs = len(all_table[ramask & decmask])
            if num_objs > 1:
                print(num_objs, ra.value)

            pos = SkyCoord(ra, dec, frame="icrs")
            cutout = Cutout2D(data, pos, stamp, wcs=wcs)

            header = cutout.wcs.to_header()
            header['NUMOBJS'] = num_objs
            for col_name, value in row.items():
                if pd.isna(value):
                    continue
                key = str(col_name)[:8].upper()
                header[key] = value
            header['BAND'] = band
            header['CATALOG'] = catalog

            folder = Path(cutout_dir / f"{float(row['RA'])}")
            folder.mkdir(parents=True, exist_ok=True)

            cutout_fits = fits.PrimaryHDU(data=cutout.data, header=header)
            cutout_fits.writeto(folder / f"{band}.fits", overwrite=True)

        hdul.close()

    size = len(list(cutout_dir.glob('*')))
    print(f"Number of cutouts: {size}")
    print("Expected: 486")

    num_bands = 3

    for folder in cutout_dir.iterdir():
        if folder.is_dir():
            count = len(list(folder.glob('*')))
            if count != num_bands:
                print(f"{folder.name}")

Set DATE-AVG to '2022-09-21T10:17:11.746' from MJD-AVG.
Set DATE-END to '2022-12-22T05:42:36.326' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -36.758463 from OBSGEO-[XYZ].
Set OBSGEO-H to 1725319218.494 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


11 214.7074428
9 214.7108799
11 214.7138002
10 214.7304391
13 214.7338924
10 214.7352097
9 214.7462083
7 214.7517458
16 214.7606245
13 214.7632297
6 214.7672274
13 214.7737591
14 214.7771113
13 214.7780825
13 214.7856892
34 214.7914919
2 214.7937745
20 214.7938251
10 214.7959339
12 214.8010535
10 214.8033616
10 214.8131564
4 214.8150258
19 214.8277359
18 214.8293075
9 214.8368571
8 214.8403407
8 214.8464674
8 214.8505883
13 214.8538728
10 214.8539018
7 214.8554077
5 214.8559449
11 214.8598201
16 214.8660438
16 214.8660523
23 214.8712338
9 214.8762902
9 214.8769415
15 214.8771133
18 214.8788841
11 214.8852052
28 214.8870681
28 214.8870721
7 214.8949122
17 214.8956165
15 214.8967058
9 214.8970339
18 214.8983291
21 214.9024287
13 214.9048498
10 214.9049662
9 214.9051029
13 214.9056056
10 214.9059755
11 214.9110583
7 214.9127277
19 214.9163617
14 214.9165073
14 214.9191048
17 214.9314778
9 214.9327959
11 214.942148
12 214.9509341
15 214.9514804
16 214.9578766
10 214.9623662
9 214.964243
10

Set DATE-AVG to '2022-09-22T03:18:18.036' from MJD-AVG.
Set DATE-END to '2022-12-25T03:57:18.887' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -36.739053 from OBSGEO-[XYZ].
Set OBSGEO-H to 1725216481.136 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


6 214.7672274
13 214.7737591
14 214.7771113
13 214.7780825
13 214.7856892
34 214.7914919
2 214.7937745
20 214.7938251
10 214.7959339
12 214.8010535
10 214.8033616
10 214.8131564
4 214.8150258
19 214.8277359
18 214.8293075
9 214.8368571
8 214.8403407
8 214.8464674
8 214.8505883
13 214.8538728
10 214.8539018
7 214.8554077
5 214.8559449
11 214.8598201
16 214.8660438
16 214.8660523
23 214.8712338
9 214.8762902
9 214.8769415
15 214.8771133
18 214.8788841
11 214.8852052
28 214.8870681
28 214.8870721
7 214.8949122
17 214.8956165
15 214.8967058
9 214.8970339
18 214.8983291
21 214.9024287
13 214.9048498
10 214.9049662
9 214.9051029
13 214.9056056
10 214.9059755
11 214.9110583
7 214.9127277
19 214.9163617
14 214.9165073
14 214.9191048
17 214.9314778
9 214.9327959
11 214.942148
12 214.9509341
15 214.9514804
16 214.9578766
10 214.9623662
9 214.964243
10 214.9711608
4 214.9741035
13 214.9761426
12 214.9774828
10 214.9785615
5 214.9805987
10 214.9818175
14 214.9892586
8 214.9934648
13 214.9950126
12

Set DATE-AVG to '2022-10-07T06:02:04.794' from MJD-AVG.
Set DATE-END to '2022-12-22T06:44:09.762' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -36.768884 from OBSGEO-[XYZ].
Set OBSGEO-H to 1725373838.960 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


11 214.7074428
9 214.7108799
11 214.7138002
10 214.7304391
13 214.7338924
10 214.7352097
9 214.7462083
7 214.7517458
16 214.7606245
13 214.7632297
6 214.7672274
13 214.7737591
14 214.7771113
13 214.7780825
13 214.7856892
34 214.7914919
2 214.7937745
20 214.7938251
10 214.7959339
12 214.8010535
10 214.8033616
10 214.8131564
4 214.8150258
19 214.8277359
18 214.8293075
9 214.8368571
8 214.8403407
8 214.8464674
8 214.8505883
13 214.8538728
10 214.8539018
7 214.8554077
5 214.8559449
11 214.8598201
16 214.8660438
16 214.8660523
23 214.8712338
9 214.8762902
9 214.8769415
15 214.8771133
18 214.8788841
11 214.8852052
28 214.8870681
28 214.8870721
7 214.8949122
17 214.8956165
15 214.8967058
9 214.8970339
18 214.8983291
21 214.9024287
13 214.9048498
10 214.9049662
9 214.9051029
13 214.9056056
10 214.9059755
11 214.9110583
7 214.9127277
19 214.9163617
14 214.9165073
14 214.9191048
17 214.9314778
9 214.9327959
11 214.942148
12 214.9509341
15 214.9514804
16 214.9578766
10 214.9623662
9 214.964243
10